In [5]:
# ============================================================
# 0. Imports
# ============================================================
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LassoCV, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    median_absolute_error,
    r2_score,
    explained_variance_score
)

from xgboost import XGBRegressor

In [6]:
# ============================================================
# 1. Load and preprocess each asset
# ============================================================

# ---- adjust paths if needed ----
path_apple = '/Users/ambervo/Library/CloudStorage/OneDrive-Personal/Documents/GMBA 3/FinTech/Group Project/data/Apple Stock Price History.csv'
path_nvda  = '/Users/ambervo/Library/CloudStorage/OneDrive-Personal/Documents/GMBA 3/FinTech/Group Project/data/NVIDIA Stock Price History.csv'
path_tsla  = '/Users/ambervo/Library/CloudStorage/OneDrive-Personal/Documents/GMBA 3/FinTech/Group Project/data/Tesla Stock Price History.csv'
path_spx   = '/Users/ambervo/Library/CloudStorage/OneDrive-Personal/Documents/GMBA 3/FinTech/Group Project/data/S&P 500 - Historical Data.csv'


def detect_close_column(df):
    """Try to guess the close/price column name."""
    candidates = ["Close", "Adj Close", "Adj Close*", "Price", "Last"]
    for c in candidates:
        if c in df.columns:
            return c
    # fallback: last column
    return df.columns[-1]


def prep_asset(df, name):
    """
    Prepare one asset:
    - parse Date
    - detect close/price column
    - compute log returns
    - compute RV20, RV60, RV120 (annualized)
    """
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

    close_col = detect_close_column(df)
    df[close_col] = (
        df[close_col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

    # log return
    df[f"{name}_logret"] = np.log(df[close_col]).diff()

    # realized volatility (annualized)
    for win in (20, 60, 120):
        df[f"{name}_RV{win}"] = np.sqrt(252) * df[f"{name}_logret"].rolling(win).std()

    keep_cols = ["Date"] + [c for c in df.columns if c.startswith(name)]
    return df[keep_cols]


apple_raw = pd.read_csv(path_apple)
nvda_raw  = pd.read_csv(path_nvda)
tsla_raw  = pd.read_csv(path_tsla)
spx_raw   = pd.read_csv(path_spx)

AAPL = prep_asset(apple_raw, "AAPL")
NVDA = prep_asset(nvda_raw,  "NVDA")
TSLA = prep_asset(tsla_raw,  "TSLA")
SPX  = prep_asset(spx_raw,   "SPX")

In [7]:
# ============================================================
# 2. Merge into one DataFrame & build features/target
# ============================================================

data = (
    SPX.merge(AAPL, on="Date")
       .merge(NVDA, on="Date")
       .merge(TSLA, on="Date")
       .sort_values("Date")
)

# Target: SPX_RV60 as realized volatility benchmark
data["y"] = data["SPX_RV60"]

# All candidate features (should be 16 total including SPX_RV60)
all_candidate_cols = [c for c in data.columns if c not in ["Date", "y"]]

# Remove target from features to get 15 predictors
feature_cols = [c for c in all_candidate_cols if c != "SPX_RV60"]

print("All candidate columns:", all_candidate_cols)
print("Final feature columns (should be 15):", feature_cols)
print("Number of features:", len(feature_cols))

# Lag features by 1 day to avoid look-ahead bias
for c in feature_cols:
    data[c] = data[c].shift(1)

# Drop rows with NaN from diff/rolling/shift
data = data.dropna().reset_index(drop=True)

X_df = data[feature_cols]
y = data["y"].values

print("Number of observations:", len(y))

All candidate columns: ['SPX_logret', 'SPX_RV20', 'SPX_RV60', 'SPX_RV120', 'AAPL_logret', 'AAPL_RV20', 'AAPL_RV60', 'AAPL_RV120', 'NVDA_logret', 'NVDA_RV20', 'NVDA_RV60', 'NVDA_RV120', 'TSLA_logret', 'TSLA_RV20', 'TSLA_RV60', 'TSLA_RV120']
Final feature columns (should be 15): ['SPX_logret', 'SPX_RV20', 'SPX_RV120', 'AAPL_logret', 'AAPL_RV20', 'AAPL_RV60', 'AAPL_RV120', 'NVDA_logret', 'NVDA_RV20', 'NVDA_RV60', 'NVDA_RV120', 'TSLA_logret', 'TSLA_RV20', 'TSLA_RV60', 'TSLA_RV120']
Number of features: 15
Number of observations: 3530


In [8]:
# ============================================================
# 3. Time-series cross-validator
# ============================================================

tscv = TimeSeriesSplit(n_splits=5)

# ============================================================
# 4. Feature Selection: LASSO (LassoFS)
# ============================================================

alphas_lasso = np.array([1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1])

lasso_fs_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso_cv", LassoCV(alphas=alphas_lasso, cv=tscv))
])

lasso_fs_pipe.fit(X_df.values, y)

lasso_cv    = lasso_fs_pipe.named_steps["lasso_cv"]
lasso_alpha = lasso_cv.alpha_          # best lambda from CV
lasso_coefs = lasso_cv.coef_

lasso_selected = [f for f, coef in zip(feature_cols, lasso_coefs)
                  if abs(coef) > 1e-6]

print("Best λ (alpha) from LASSO CV:", lasso_alpha)
print("LASSO-selected features (LassoFS):", lasso_selected)

Best λ (alpha) from LASSO CV: 0.0001
LASSO-selected features (LassoFS): ['SPX_logret', 'SPX_RV20', 'SPX_RV120', 'AAPL_logret', 'AAPL_RV20', 'AAPL_RV60', 'AAPL_RV120', 'NVDA_logret', 'NVDA_RV20', 'NVDA_RV60', 'NVDA_RV120', 'TSLA_RV20', 'TSLA_RV60', 'TSLA_RV120']


In [9]:
# ============================================================
# Display LASSO Coefficients (for academic interpretation)
# ============================================================

lasso_coef_table = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": lasso_coefs
}).sort_values("Coefficient", ascending=False)

print("\n===== LASSO Coefficients (sorted) =====")
print(lasso_coef_table)

# Non-zero coefficients only (more relevant for interpretation)
print("\n===== Non-zero LASSO Coefficients =====")
print(lasso_coef_table[lasso_coef_table["Coefficient"].abs() > 1e-6])


===== LASSO Coefficients (sorted) =====
        Feature  Coefficient
2     SPX_RV120     0.050238
5     AAPL_RV60     0.038690
1      SPX_RV20     0.030828
9     NVDA_RV60     0.025261
13    TSLA_RV60     0.006186
3   AAPL_logret     0.000395
7   NVDA_logret     0.000224
11  TSLA_logret     0.000000
0    SPX_logret    -0.000451
12    TSLA_RV20    -0.001880
14   TSLA_RV120    -0.004491
8     NVDA_RV20    -0.008106
10   NVDA_RV120    -0.014929
4     AAPL_RV20    -0.017204
6    AAPL_RV120    -0.021576

===== Non-zero LASSO Coefficients =====
        Feature  Coefficient
2     SPX_RV120     0.050238
5     AAPL_RV60     0.038690
1      SPX_RV20     0.030828
9     NVDA_RV60     0.025261
13    TSLA_RV60     0.006186
3   AAPL_logret     0.000395
7   NVDA_logret     0.000224
0    SPX_logret    -0.000451
12    TSLA_RV20    -0.001880
14   TSLA_RV120    -0.004491
8     NVDA_RV20    -0.008106
10   NVDA_RV120    -0.014929
4     AAPL_RV20    -0.017204
6    AAPL_RV120    -0.021576


In [10]:
# ============================================================
# 5. Feature Selection: XGBoost importance (XGBFS)
# ============================================================

xgb_fs = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb_fs.fit(X_df.values, y)
importances = xgb_fs.feature_importances_

# choose top-k important features (e.g. top 10)
k = min(10, len(feature_cols))
idx_sorted = np.argsort(importances)[::-1]
xgb_selected = [feature_cols[i] for i in idx_sorted[:k]]

print("Top XGB-selected features (XGBFS):", xgb_selected)

Top XGB-selected features (XGBFS): ['SPX_RV120', 'AAPL_RV60', 'TSLA_RV60', 'SPX_RV20', 'NVDA_RV60', 'AAPL_RV120', 'TSLA_RV120', 'NVDA_RV120', 'AAPL_RV20', 'TSLA_RV20']


In [11]:
# ============================================================
# Display XGBoost Feature Importance (for academic interpretation)
# ============================================================

xgb_importance_table = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importances
}).sort_values("Importance", ascending=False)

print("\n===== XGBoost Feature Importances (sorted) =====")
print(xgb_importance_table)


===== XGBoost Feature Importances (sorted) =====
        Feature  Importance
2     SPX_RV120    0.320350
5     AAPL_RV60    0.209787
13    TSLA_RV60    0.198566
1      SPX_RV20    0.181842
9     NVDA_RV60    0.039780
6    AAPL_RV120    0.016187
14   TSLA_RV120    0.012384
10   NVDA_RV120    0.009186
4     AAPL_RV20    0.004062
12    TSLA_RV20    0.003162
8     NVDA_RV20    0.003096
0    SPX_logret    0.000604
3   AAPL_logret    0.000566
7   NVDA_logret    0.000255
11  TSLA_logret    0.000173


In [12]:
# ============================================================
# 6. Define Feature-Selection Strategies
#    - AllFS   : all 15 features (benchmark)
#    - LassoFS : LASSO-selected features
#    - XGBFS   : XGBoost-selected features
# ============================================================

fs_dict = {
    "AllFS":   feature_cols,   # benchmark
    "LassoFS": lasso_selected,
    "XGBFS":   xgb_selected
}

In [13]:

# ============================================================
# 7. Model factory (4 ML models)
# ============================================================

def make_model(name, lasso_alpha=None):
    if name == "LinearRegression":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ])

    elif name == "Lasso":
        alpha = lasso_alpha if lasso_alpha is not None else 1e-3
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", Lasso(alpha=alpha))
        ])

    elif name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1
        )

    elif name == "XGBoost":
        return XGBRegressor(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
    else:
        raise ValueError(f"Unknown model {name}")


models = ["LinearRegression", "Lasso", "RandomForest", "XGBoost"]

In [14]:
# ============================================================
# 8. Metrics function
# ============================================================

def compute_metrics(y_true, y_pred):
    """Compute all metrics for one fold."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    evs  = explained_variance_score(y_true, y_pred)

    corr = np.corrcoef(y_true, y_pred)[0, 1]

    # QLIKE (ensure positivity)
    eps = 1e-8
    y_true_pos = np.clip(y_true, eps, None)
    y_pred_pos = np.clip(y_pred, eps, None)
    ratio = y_true_pos / y_pred_pos
    qlike = np.mean(ratio - np.log(ratio) - 1)

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "MedAE": medae,
        "R2": r2,
        "ExplVar": evs,
        "Corr": corr,
        "QLIKE": qlike
    }


In [15]:
# ============================================================
# 9. Time-series CV: all model × FS combinations
#    (includes benchmark: LinearRegression + AllFS)
# ============================================================

results = []

for fs_name, feat_list in fs_dict.items():
    X_sel = X_df[feat_list].values

    for mname in models:
        model = make_model(mname, lasso_alpha=lasso_alpha)

        # metrics for each fold
        fold_metrics = {k: [] for k in
                        ["RMSE", "MAE", "MAPE", "MedAE",
                         "R2", "ExplVar", "Corr", "QLIKE"]}

        for train_idx, test_idx in tscv.split(X_sel):
            X_train, X_test = X_sel[train_idx], X_sel[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            m = compute_metrics(y_test, y_pred)
            for key in fold_metrics.keys():
                fold_metrics[key].append(m[key])

        # aggregate across folds
        summary = {
            "FeatureSelection": fs_name,
            "Model": mname,
            "n_features": len(feat_list)
        }
        for key, vals in fold_metrics.items():
            summary[f"{key}_mean"] = np.mean(vals)
            summary[f"{key}_std"]  = np.std(vals)

        results.append(summary)

results_df = pd.DataFrame(results)

# nice column ordering
cols_order = [
    "FeatureSelection", "Model", "n_features",
    "RMSE_mean", "RMSE_std",
    "MAE_mean", "MAE_std",
    "MAPE_mean", "MAPE_std",
    "MedAE_mean", "MedAE_std",
    "R2_mean", "R2_std",
    "ExplVar_mean", "ExplVar_std",
    "Corr_mean", "Corr_std",
    "QLIKE_mean", "QLIKE_std"
]

results_df = results_df[cols_order].sort_values(
    ["FeatureSelection", "RMSE_mean"]
)

# Show full table (optional)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

print(results_df.round(4))

# Save to CSV for visualization step
results_df.round(6).to_csv('/Users/ambervo/Library/CloudStorage/OneDrive-Personal/Documents/GMBA 3/FinTech/Group Project/Trading Strategy Lokia Amber/Testing Result(3).csv', index=False)

   FeatureSelection             Model  n_features  RMSE_mean  RMSE_std  \
1             AllFS             Lasso          15     0.0263    0.0094   
0             AllFS  LinearRegression          15     0.0267    0.0101   
3             AllFS           XGBoost          15     0.0374    0.0201   
2             AllFS      RandomForest          15     0.0430    0.0193   
5           LassoFS             Lasso          14     0.0263    0.0094   
4           LassoFS  LinearRegression          14     0.0267    0.0101   
7           LassoFS           XGBoost          14     0.0373    0.0199   
6           LassoFS      RandomForest          14     0.0428    0.0191   
9             XGBFS             Lasso          10     0.0271    0.0097   
8             XGBFS  LinearRegression          10     0.0277    0.0106   
11            XGBFS           XGBoost          10     0.0374    0.0195   
10            XGBFS      RandomForest          10     0.0424    0.0194   

    MAE_mean  MAE_std  MAPE_mean  MAP